In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

c:\Users\rahul\sql_ai_agent\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

model = ChatOpenAI(model='gpt-4o-mini')

In [3]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description="Sentiment of the review")

In [16]:
class DiagnosticSchema(BaseModel):
    issue_type: Literal["UX","Performance","Bug","Support","Other"] = Field(description="The category of issue mentioned in the review")
    tone: Literal["angry","frustrated","disappointed","calm"] = Field(description="The emotional tone used by the user")
    urgency: Literal["high", "medium","low"] = Field(description="How urgent or critical the issue appears to be")

In [17]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model_2 = model.with_structured_output(DiagnosticSchema)

In [18]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive","negative"]
    response: str
    diagnostic: dict

In [19]:
def find_sentiment(state:ReviewState):
    prompt = f"For the following review, find out the sentiment\n{state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment':sentiment}

In [20]:
def check_sentiment(state:ReviewState)->Literal["positive_response","diagnostics"]:
    if state["sentiment"]=="positive":
        return "positive_response"
    elif state["sentiment"]=="negative":
        return "diagnostics"

In [21]:
def positive_response(state:ReviewState):
    prompt = f"For the following positive review, write a warm thankyou message in response.\n{state['review']}\n\nAlso ask the reviewr to leave a feedback on the website"
    response = model.invoke(prompt).content

    return {'response':response}

In [22]:
def diagnostics(state:ReviewState):
    prompt = f"Diagnose the input_type, tone and urgency of this negative review.\n{state['review']}"
    diagnostic = structured_model_2.invoke(prompt).model_dump()

    return {'diagnostic':diagnostic}

In [23]:
def negative_response(state:ReviewState):
    diagnostic = state['diagnostic']
    prompt = f"""You are a support assistant. The user had a {diagnostic['issue_type']} issue, sounded {diagnostic['tone']} and marked urgency as {diagnostic['urgency']}. Write an empathetic, helpful resolution message."""
    response = model.invoke(prompt).content

    return {'response':response}

In [24]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('diagnostics',diagnostics)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('diagnostics','negative_response')
graph.add_edge('negative_response',END)

workflow = graph.compile()

In [25]:
initial_state = {'review':"I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."}
final_state = workflow.invoke(initial_state)

final_state

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'response': "Subject: We're Here to Help with Your Bug Issue\n\nDear [User's Name],\n\nI truly understand how frustrating it can be to encounter a bug, especially when you're trying to accomplish something important. Your urgency is perfectly valid, and I’m here to support you in resolving this as quickly as possible.\n\nTo assist you better, could you please provide me with a bit more detail about the issue you’re facing? Specifically, if you could share the steps that led to the bug, any error messages you received, and the environment you're using (such as the device, operating system, or app version), it would help us identify the root cause more effectively.\n\nIn the meantime, please rest assured that we are comm